# 05. Thresholding and morphology

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner

## What you will learn
- Create binary masks
- Know global versus local thresholding
- Choose foreground polarity
- Use morphology conservatively

> **Learning rule:** understand the problem first, then choose the function.

## 1. Thresholding converts intensity into a decision

A threshold mask is `True` for selected pixels and `False` elsewhere. It is useful only when intensity meaningfully separates target from background.

In [ ]:
import matplotlib.pyplot as plt
import skimage as ski
from skimage import filters, morphology
image = ski.data.coins()
# Otsu estimates one global threshold from the histogram.
t = filters.threshold_otsu(image)
# Use > for bright foreground; use < when the target is darker.
mask = image > t
print("threshold:", t, "foreground fraction:", mask.mean())

## 2. Global and local thresholds

Otsu gives one threshold for the whole image. Local thresholding is useful when background or illumination varies across the field.

In [ ]:
# Local threshold returns a threshold IMAGE rather than one number.
local_t = filters.threshold_local(image, block_size=51, offset=5)
local_mask = image > local_t
fig, ax=plt.subplots(1,3,figsize=(12,4))
for a,im,title in zip(ax,[image,mask,local_mask],["Image","Global Otsu","Local threshold"]):
    a.imshow(im,cmap="gray"); a.set_title(title); a.axis("off")
plt.show()

## 3. Morphology cleans masks but changes geometry

Use size-based cleanup only when small components/holes are known artifacts. Record the size parameter and remember it is in pixels unless calibrated.

In [ ]:
# scikit-image 0.26: max_size removes/fills components up to this pixel area.
clean = morphology.remove_small_objects(mask, max_size=60)
clean = morphology.remove_small_holes(clean, max_size=60)
fig, ax=plt.subplots(1,2,figsize=(8,4))
ax[0].imshow(mask,cmap="gray"); ax[0].set_title("Raw mask")
ax[1].imshow(clean,cmap="gray"); ax[1].set_title("Cleaned mask")
for a in ax: a.axis("off")
plt.show()

## Function-selection guide

| Situation | Start with | Why |
|---|---|---|
| One global split is plausible | `threshold_otsu()` | Simple global baseline |
| Illumination varies | `threshold_local()` | Spatially varying decision |
| Tiny foreground artifacts | `remove_small_objects()` | Size-based cleanup |
| Tiny internal holes | `remove_small_holes()` | Fill selected holes |

**Always overlay/compare the mask with the raw image before measuring.**

## Takeaway

**Choose functions because they solve a specific image problem, and always inspect the result before measuring.**